# Flow Matching for Generative Modeling
## 论文完整精读笔记：数学基础、公式推导、算法原理与应用

**论文信息**

- **论文名称**：Flow Matching for Generative Modeling
- **作者**：Yaron Lipman, Ricky T. Q. Chen, Hannes Ben-Hamu, Maximilian Nickel, Matt Le
- **发表会议**：ICLR 2023
- **论文地址**：https://arxiv.org/pdf/2210.02747
- **研究方向**：生成模型、Continuous Normalizing Flow、Flow Matching、Optimal Transport

**笔记约定：**

1. 所有公式按照章节连续编号，例如式 (1.1)、式 (1.2)、式 (2.1)。
2. 数学推导明确指出使用了哪些公式，以及代入、求导、积分和整理的过程。
3. 变量首次出现时说明其数学含义。
4. 区分条件概率路径、边缘概率路径、条件速度场和边缘速度场。
5. 公式编号为本笔记独立编号，并非论文原始编号。
6. 首先介绍必要的数学基础，再按照论文的理论逻辑展开。

---

# 第一章：Flow Matching 的整体框架

## 1.1 生成模型的目标

生成模型的目标是学习真实数据的概率分布，并从中生成新的数据样本。

假设真实数据随机变量为：

$$
Y\sim q(y)
\tag{1.1}
$$

其中：

- $Y\in\mathbb R^d$：真实数据随机变量。
- $y$：随机变量 $Y$ 的具体取值。
- $q(y)$：真实数据的概率密度。
- $d$：数据维度。

例如，在自动驾驶轨迹规划中，一条未来轨迹可以表示为：

$$
Y=
(x_1,y_1,x_2,y_2,\ldots,x_T,y_T)
\tag{1.2}
$$

其中 $T$ 是未来轨迹的物理时间帧数。

如果轨迹只包含二维坐标，则：

$$
Y\in\mathbb R^{2T}
$$

生成模型希望从一个简单的初始分布出发，例如：

$$
Z\sim\mathcal N(0,I)
\tag{1.3}
$$

经过某种变换得到：

$$
X_1\sim q
\tag{1.4}
$$

因此，生成模型需要学习一个映射：

$$
\boxed{
Z\longrightarrow Y
}
\tag{1.5}
$$

Flow Matching 通过连续时间的运动过程实现这一映射。

---

## 1.2 Continuous Normalizing Flow

Continuous Normalizing Flow，简称 CNF，是一种基于常微分方程的生成模型。

定义：

$$
\boxed{
\frac{dX_t}{dt}=v_\theta(X_t,t)
}
\tag{1.6}
$$

其中：

- $t\in[0,1]$：生成模型的时间变量。
- $X_t\in\mathbb R^d$：时间 $t$ 的中间状态。
- $v_\theta(x,t)\in\mathbb R^d$：神经网络预测的速度场。
- $\theta$：神经网络参数。

初始条件：

$$
X_0=Z,\qquad Z\sim\mathcal N(0,I)
\tag{1.7}
$$

通过求解式 (1.6)，得到：

$$
X_1
=
X_0+
\int_0^1v_\theta(X_t,t)\,dt
\tag{1.8}
$$

希望最终：

$$
X_1\sim q
\tag{1.9}
$$

整个生成过程为：

$$
\boxed{
\mathcal N(0,I)
\xrightarrow{\text{ODE}}
q
}
\tag{1.10}
$$

### Flow Matching 的核心问题

CNF 需要训练速度网络：

$$
v_\theta(x,t)
$$

但是：

**怎样知道在任意位置 $x$、任意时间 $t$，正确的速度应该是多少？**

Flow Matching 的解决方法是：

先构造一条目标概率路径：

$$
p_t(x)
$$

再构造能够生成该概率路径的目标速度场：

$$
u_t(x)
$$

然后使用均方误差训练：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E_{t,X_t\sim p_t}
\left[
\|v_\theta(X_t,t)-u_t(X_t)\|^2
\right]
}
\tag{1.11}
$$

但边缘速度场 $u_t(x)$ 通常无法直接计算。

因此，论文进一步提出 Conditional Flow Matching，通过容易计算的条件速度场完成训练。

---

# 第二章：数学基础——随机变量变换与概率路径

## 2.1 随机变量与概率分布

假设：

$$
Z\sim p_Z(z)
\tag{2.1}
$$

通过确定性函数：

$$
X=f(Z)
\tag{2.2}
$$

得到新的随机变量 $X$。

需要区分两个对象：

**随机变量变换：**

$$
Z\mapsto X
$$

**概率分布变换：**

$$
p_Z\mapsto p_X
$$

随机变量经过确定性映射，其概率分布也会随之发生变化。

---

## 2.2 一维变量替换公式

假设：

$$
X=f(Z)
\tag{2.3}
$$

其中 $f$ 可逆且可微。

考虑随机变量 $Z$ 在小区间：

$$
[z,z+dz]
$$

中的概率质量：

$$
P(Z\in[z,z+dz])
\approx p_Z(z)|dz|
\tag{2.4}
$$

经过映射 $f$ 后，该区间对应：

$$
[x,x+dx]
$$

其概率质量为：

$$
P(X\in[x,x+dx])
\approx p_X(x)|dx|
\tag{2.5}
$$

由于确定性映射不会改变这部分概率质量，将式 (2.4) 与式 (2.5) 相等：

$$
p_Z(z)|dz|
=
p_X(x)|dx|
\tag{2.6}
$$

两侧除以 $|dx|$：

$$
p_X(x)
=
p_Z(z)
\left|\frac{dz}{dx}\right|
\tag{2.7}
$$

根据式 (2.3)：

$$
z=f^{-1}(x)
\tag{2.8}
$$

**将式 (2.8) 代入式 (2.7)：**

$$
\boxed{
p_X(x)
=
p_Z(f^{-1}(x))
\left|
\frac{df^{-1}(x)}{dx}
\right|
}
\tag{2.9}
$$

这就是概率论中的变量替换公式。

---

## 2.3 高维变量替换公式

当：

$$
Z,X\in\mathbb R^d
$$

时，使用 Jacobian 矩阵描述局部体积变化。

定义：

$$
J_f(z)
=
\frac{\partial f(z)}{\partial z}
\tag{2.10}
$$

则：

$$
\boxed{
p_X(x)
=
p_Z(f^{-1}(x))
\left|
\det J_{f^{-1}}(x)
\right|
}
\tag{2.11}
$$

其中：

$$
\det J_{f^{-1}}(x)
$$

描述逆映射对局部体积的缩放比例。

例如二维情况下：

$$
X=2Z
\tag{2.12}
$$

对应逆映射：

$$
Z=\frac X2
$$

因此：

$$
J_{f^{-1}}
=
\begin{bmatrix}
1/2&0\\
0&1/2
\end{bmatrix}
\tag{2.13}
$$

行列式：

$$
\det J_{f^{-1}}=\frac14
\tag{2.14}
$$

**将式 (2.14) 代入式 (2.11)：**

$$
p_X(x)
=
\frac14p_Z(x/2)
\tag{2.15}
$$

这说明二维空间面积扩大四倍后，相应位置的概率密度降低四倍。

---

## 2.4 Pushforward：概率分布的推前

假设：

$$
X_0\sim p_0
\tag{2.16}
$$

定义一个时间相关的映射：

$$
X_t=\phi_t(X_0)
\tag{2.17}
$$

则 $X_t$ 的概率分布称为 $p_0$ 在映射 $\phi_t$ 下的推前分布：

$$
\boxed{
p_t=(\phi_t)_\#p_0
}
\tag{2.18}
$$

其中：

$$
(\phi_t)_\#
$$

是 Pushforward 算子。

严格地说，对于任意可测集合 $A$：

$$
P(X_t\in A)
=
P(X_0\in\phi_t^{-1}(A))
\tag{2.19}
$$

式 (2.19) 是推前分布的定义，不要求映射处处具有密度表达式。

如果映射可逆且可微，则可以进一步使用式 (2.11) 计算概率密度。

---

## 2.5 概率路径与样本轨迹

定义一族随机变量：

$$
\{X_t\}_{t\in[0,1]}
\tag{2.20}
$$

每个时间对应：

$$
X_t\sim p_t
\tag{2.21}
$$

于是：

$$
\boxed{
t\mapsto p_t
}
\tag{2.22}
$$

称为概率路径。

如果固定初始样本：

$$
X_0=x_0
$$

则：

$$
\boxed{
t\mapsto\phi_t(x_0)
}
\tag{2.23}
$$

称为样本轨迹。

两者具有不同含义：

| 对象 | 数学表达 | 含义 |
|---|---|---|
| 概率路径 | $p_t$ | 整个概率分布随时间变化 |
| 样本轨迹 | $x_t$ | 一个粒子随时间运动 |
| 速度场 | $v_t(x)$ | 空间每个位置的运动速度 |
| 流映射 | $\phi_t(x_0)$ | 初始粒子到当前粒子位置的映射 |

**重要结论：**

给定概率路径 $p_t$，并不能唯一确定每个粒子的运动轨迹。

不同的速度场可能生成相同的概率路径。

---

# 第三章：连续性方程

## 3.1 连续性方程的物理意义

连续性方程用于描述概率质量守恒。

假设一维空间中，粒子的概率密度为：

$$
p_t(x)
$$

运动速度为：

$$
u_t(x)
$$

定义概率流量：

$$
\boxed{
j_t(x)=p_t(x)u_t(x)
}
\tag{3.1}
$$

概率流量表示单位时间内穿过某个位置的概率质量。

---

## 3.2 一维连续性方程推导

考虑区间：

$$
[a,b]
$$

区间内的总概率质量：

$$
M(t)=\int_a^bp_t(x)\,dx
\tag{3.2}
$$

区间内概率质量的变化等于左侧流入减去右侧流出：

$$
\frac{dM(t)}{dt}
=
j_t(a)-j_t(b)
\tag{3.3}
$$

**将式 (3.2) 代入式 (3.3)：**

$$
\frac{d}{dt}
\int_a^bp_t(x)\,dx
=
j_t(a)-j_t(b)
\tag{3.4}
$$

左侧将时间微分移入积分：

$$
\int_a^b
\frac{\partial p_t(x)}{\partial t}
dx
=
j_t(a)-j_t(b)
\tag{3.5}
$$

根据微积分基本定理：

$$
j_t(b)-j_t(a)
=
\int_a^b
\frac{\partial j_t(x)}{\partial x}dx
\tag{3.6}
$$

**将式 (3.6) 代入式 (3.5)：**

$$
\int_a^b
\frac{\partial p_t(x)}{\partial t}dx
=
-\int_a^b
\frac{\partial j_t(x)}{\partial x}dx
\tag{3.7}
$$

移项：

$$
\int_a^b
\left[
\frac{\partial p_t(x)}{\partial t}
+
\frac{\partial j_t(x)}{\partial x}
\right]dx
=0
\tag{3.8}
$$

由于区间 $[a,b]$ 任意，在适当正则条件下：

$$
\boxed{
\frac{\partial p_t(x)}{\partial t}
+
\frac{\partial j_t(x)}{\partial x}
=0
}
\tag{3.9}
$$

**将式 (3.1) 代入式 (3.9)：**

$$
\boxed{
\frac{\partial p_t(x)}{\partial t}
+
\frac{\partial}{\partial x}
[p_t(x)u_t(x)]
=0
}
\tag{3.10}
$$

这就是一维连续性方程。

---

## 3.3 高维连续性方程

对于 $d$ 维空间：

$$
x\in\mathbb R^d
$$

定义速度场：

$$
u_t(x)\in\mathbb R^d
$$

概率流量：

$$
j_t(x)=p_t(x)u_t(x)
\tag{3.11}
$$

将式 (3.10) 推广到高维：

$$
\boxed{
\frac{\partial p_t(x)}{\partial t}
+
\nabla_x\cdot[p_t(x)u_t(x)]
=0
}
\tag{3.12}
$$

其中：

$$
\nabla_x\cdot j_t(x)
=
\sum_{i=1}^{d}
\frac{\partial j_{t,i}(x)}{\partial x_i}
\tag{3.13}
$$

称为散度。

连续性方程表达：

$$
\boxed{
\text{局部概率密度变化}
=
-\text{局部概率流量的净流出}
}
\tag{3.14}
$$

---

## 3.4 连续性方程与 ODE 的关系

假设粒子满足：

$$
\boxed{
\frac{dX_t}{dt}=u_t(X_t)
}
\tag{3.15}
$$

初始分布：

$$
X_0\sim p_0
\tag{3.16}
$$

在适当正则条件下，式 (3.15) 生成的概率分布满足：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tu_t)=0
}
\tag{3.17}
$$

因此：

$$
\boxed{
\text{ODE 描述粒子运动}
}
$$

$$
\boxed{
\text{连续性方程描述粒子群的概率分布演化}
}
$$

两者是同一动力学过程的不同数学描述。

---

## 3.5 连续性方程是否唯一决定速度场？

不一定。

假设：

$$
\partial_tp_t+\nabla\cdot(p_tu_t)=0
\tag{3.18}
$$

构造另一个速度场：

$$
\widetilde u_t
=
u_t+\frac{w_t}{p_t}
\tag{3.19}
$$

其中在 $p_t>0$ 的区域定义该式，并假设：

$$
\nabla\cdot w_t=0
\tag{3.20}
$$

计算新的概率流量：

$$
p_t\widetilde u_t
=
p_tu_t+w_t
\tag{3.21}
$$

两侧取散度：

$$
\nabla\cdot(p_t\widetilde u_t)
=
\nabla\cdot(p_tu_t)+\nabla\cdot w_t
\tag{3.22}
$$

**将式 (3.20) 代入式 (3.22)：**

$$
\nabla\cdot(p_t\widetilde u_t)
=
\nabla\cdot(p_tu_t)
\tag{3.23}
$$

因此：

$$
\partial_tp_t+\nabla\cdot(p_t\widetilde u_t)=0
\tag{3.24}
$$

说明两个不同速度场可以生成相同的概率路径。

**结论：**

$$
\boxed{
\text{概率路径不一定唯一决定速度场}
}
\tag{3.25}
$$

这也是后续区分条件速度与边缘速度的重要基础。

---

# 第四章：CNF 与 Flow Matching

## 4.1 CNF 的生成过程

定义神经网络速度场：

$$
v_\theta(x,t)
$$

生成过程满足：

$$
\boxed{
\frac{dX_t}{dt}=v_\theta(X_t,t)
}
\tag{4.1}
$$

初始：

$$
X_0\sim p_0=\mathcal N(0,I)
\tag{4.2}
$$

目标：

$$
X_1\sim q
\tag{4.3}
$$

对应概率路径：

$$
p_t=(\phi_t)_\#p_0
\tag{4.4}
$$

---

## 4.2 CNF 的瞬时变量替换公式

根据连续性方程：

$$
\partial_tp_t+\nabla\cdot(p_tv_t)=0
\tag{4.5}
$$

利用散度乘积法则：

$$
\nabla\cdot(p_tv_t)
=
v_t\cdot\nabla p_t
+
p_t\nabla\cdot v_t
\tag{4.6}
$$

**将式 (4.6) 代入式 (4.5)：**

$$
\partial_tp_t
+
v_t\cdot\nabla p_t
+
p_t\nabla\cdot v_t
=0
\tag{4.7}
$$

沿着 ODE 粒子轨迹：

$$
\dot X_t=v_t(X_t)
\tag{4.8}
$$

根据链式法则：

$$
\frac{d}{dt}p_t(X_t)
=
\partial_tp_t(X_t)
+
\nabla p_t(X_t)^\top\dot X_t
\tag{4.9}
$$

**将式 (4.8) 代入式 (4.9)：**

$$
\frac{d}{dt}p_t(X_t)
=
\partial_tp_t(X_t)
+
\nabla p_t(X_t)^\top v_t(X_t)
\tag{4.10}
$$

根据式 (4.7)：

$$
\partial_tp_t+v_t\cdot\nabla p_t
=
-p_t\nabla\cdot v_t
\tag{4.11}
$$

**将式 (4.11) 代入式 (4.10)：**

$$
\frac{d}{dt}p_t(X_t)
=
-p_t(X_t)\nabla\cdot v_t(X_t)
\tag{4.12}
$$

两侧除以 $p_t(X_t)$：

$$
\frac{1}{p_t(X_t)}
\frac{d}{dt}p_t(X_t)
=
-\nabla\cdot v_t(X_t)
\tag{4.13}
$$

根据对数求导公式：

$$
\frac{d}{dt}\log p_t(X_t)
=
\frac{1}{p_t(X_t)}
\frac{d}{dt}p_t(X_t)
\tag{4.14}
$$

**将式 (4.14) 代入式 (4.13)：**

$$
\boxed{
\frac{d}{dt}\log p_t(X_t)
=
-\nabla\cdot v_t(X_t)
}
\tag{4.15}
$$

对式 (4.15) 从 $t=0$ 积分到 $t=1$：

$$
\log p_1(X_1)-\log p_0(X_0)
=
-\int_0^1\nabla\cdot v_t(X_t)\,dt
$$

整理：

$$
\boxed{
\log p_1(X_1)
=
\log p_0(X_0)
-
\int_0^1\nabla\cdot v_t(X_t)\,dt
}
\tag{4.16}
$$

这就是 CNF 的瞬时变量替换公式。

它是第二章变量替换公式在连续时间下的对应形式。

---

## 4.3 Flow Matching 的目标

假设已经构造目标概率路径：

$$
p_t(x)
\tag{4.17}
$$

以及能够生成它的目标速度场：

$$
u_t(x)
\tag{4.18}
$$

满足：

$$
\partial_tp_t+\nabla\cdot(p_tu_t)=0
\tag{4.19}
$$

训练网络：

$$
v_\theta(x,t)
$$

逼近：

$$
u_t(x)
$$

定义损失：

$$
\boxed{
\mathcal L_{\mathrm{FM}}(\theta)
=
\mathbb E_{t\sim U(0,1)}
\mathbb E_{X_t\sim p_t}
\left[
\|v_\theta(X_t,t)-u_t(X_t)\|^2
\right]
}
\tag{4.20}
$$

如果：

$$
v_\theta(x,t)=u_t(x)
\tag{4.21}
$$

那么根据式 (4.19)，模型生成的概率路径就是目标概率路径。

### 直接训练的困难

式 (4.20) 需要知道：

$$
u_t(x)
$$

但边缘速度场通常无法直接计算。

因此需要将训练目标转换为容易计算的条件速度场。

---

# 第五章：条件概率路径与边缘概率路径

## 5.1 条件概率路径

设真实数据：

$$
Y\sim q(y)
\tag{5.1}
$$

对于固定的真实数据样本：

$$
Y=y
$$

构造条件概率路径：

$$
\boxed{
p_t(x\mid y)
}
\tag{5.2}
$$

例如选择高斯条件路径：

$$
p_t(x\mid y)
=
\mathcal N(x;\mu_t(y),\sigma_t(y)^2I)
\tag{5.3}
$$

其中：

- $\mu_t(y)$：条件高斯均值。
- $\sigma_t(y)$：条件高斯标准差。
- $y$：固定的真实数据样本。

---

## 5.2 边缘概率路径

由于：

$$
Y\sim q
$$

且：

$$
X_t\mid Y=y\sim p_t(x\mid y)
$$

联合概率密度为：

$$
p_t(x,y)=p_t(x\mid y)q(y)
\tag{5.4}
$$

对 $y$ 积分：

$$
p_t(x)
=
\int p_t(x,y)\,dy
\tag{5.5}
$$

**将式 (5.4) 代入式 (5.5)：**

$$
\boxed{
p_t(x)
=
\int p_t(x\mid y)q(y)\,dy
}
\tag{5.6}
$$

这就是边缘概率路径。

它表示将所有条件概率路径按照真实数据分布进行混合。

---

## 5.3 高斯条件路径与高斯卷积

假设：

$$
p_t(x\mid y)
=
\mathcal N(x;\alpha_ty,\sigma_t^2I)
\tag{5.7}
$$

根据式 (5.6)：

$$
p_t(x)
=
\int
\mathcal N(x;\alpha_ty,\sigma_t^2I)
q(y)\,dy
\tag{5.8}
$$

引入随机变量：

$$
Z\sim\mathcal N(0,I)
\tag{5.9}
$$

则：

$$
\boxed{
X_t=\alpha_tY+\sigma_tZ
}
\tag{5.10}
$$

具有式 (5.8) 所定义的边缘分布。

如果定义：

$$
\widetilde Y_t=\alpha_tY
\tag{5.11}
$$

那么：

$$
X_t=\widetilde Y_t+\sigma_tZ
\tag{5.12}
$$

因此：

$$
\boxed{
p_t
=
q_{\alpha_tY}
*
\mathcal N(0,\sigma_t^2I)
}
\tag{5.13}
$$

其中：

$$
q_{\alpha_tY}
$$

表示缩放后真实数据随机变量的分布。

这说明边缘概率路径可以理解为：

**缩放后的真实数据分布与高斯噪声分布进行卷积。**

需要注意，当 $\alpha_t\neq1$ 时，不能简单写成原始 $q$ 与高斯分布的卷积，而应考虑真实数据分布的缩放。

---

## 5.4 时间与概率路径的关系

固定：

$$
t=t_0
$$

得到：

$$
p_{t_0}(x)
$$

这是一个概率分布。

当：

$$
t:0\rightarrow1
$$

连续变化时，得到：

$$
p_0\rightarrow p_t\rightarrow p_1
$$

这才构成概率路径。

同理，固定：

$$
Y=y
$$

可以得到一条条件概率路径：

$$
p_t(x\mid y)
$$

对 $Y$ 边缘化，则得到边缘概率路径：

$$
p_t(x)
$$

---

# 第六章：条件速度与边缘速度

这是整篇论文最重要的理论部分。

## 6.1 条件连续性方程

假设对于每个固定：

$$
Y=y
$$

存在条件速度场：

$$
u_t(x\mid y)
\tag{6.1}
$$

能够生成条件概率路径：

$$
p_t(x\mid y)
$$

因此满足：

$$
\boxed{
\partial_tp_t(x\mid y)
+
\nabla\cdot
[p_t(x\mid y)u_t(x\mid y)]
=0
}
\tag{6.2}
$$

定义条件概率流量：

$$
\boxed{
j_t(x\mid y)
=
p_t(x\mid y)u_t(x\mid y)
}
\tag{6.3}
$$

---

## 6.2 对条件连续性方程积分

根据边缘概率公式：

$$
p_t(x)
=
\int p_t(x\mid y)q(y)\,dy
\tag{6.4}
$$

对式 (6.4) 关于 $t$ 求导：

$$
\partial_tp_t(x)
=
\partial_t
\int p_t(x\mid y)q(y)\,dy
\tag{6.5}
$$

由于 $q(y)$ 不依赖生成时间 $t$，在允许交换微分与积分的条件下：

$$
\boxed{
\partial_tp_t(x)
=
\int
\partial_tp_t(x\mid y)q(y)\,dy
}
\tag{6.6}
$$

根据条件连续性方程 (6.2)：

$$
\partial_tp_t(x\mid y)
=
-\nabla\cdot
[p_t(x\mid y)u_t(x\mid y)]
\tag{6.7}
$$

**将式 (6.7) 代入式 (6.6)：**

$$
\partial_tp_t(x)
=
-\int
\nabla\cdot
[p_t(x\mid y)u_t(x\mid y)]
q(y)\,dy
\tag{6.8}
$$

由于散度是对 $x$ 求导，积分是对 $y$ 积分，因此在适当正则条件下可以交换运算：

$$
\boxed{
\partial_tp_t(x)
=
-\nabla\cdot
\left[
\int
p_t(x\mid y)u_t(x\mid y)q(y)\,dy
\right]
}
\tag{6.9}
$$

---

## 6.3 构造边缘概率流量

定义：

$$
\boxed{
j_t(x)
=
\int
p_t(x\mid y)u_t(x\mid y)q(y)\,dy
}
\tag{6.10}
$$

**将式 (6.10) 代入式 (6.9)：**

$$
\boxed{
\partial_tp_t(x)+\nabla\cdot j_t(x)=0
}
\tag{6.11}
$$

说明式 (6.10) 构造的概率流量满足边缘概率质量守恒。

根据概率流量定义：

$$
j_t(x)=p_t(x)u_t(x)
\tag{6.12}
$$

**将式 (6.12) 代入式 (6.10)：**

$$
p_t(x)u_t(x)
=
\int
p_t(x\mid y)u_t(x\mid y)q(y)\,dy
\tag{6.13}
$$

两侧除以：

$$
p_t(x)
$$

得到：

$$
\boxed{
u_t(x)
=
\frac{
\int p_t(x\mid y)u_t(x\mid y)q(y)\,dy
}{
p_t(x)
}
}
\tag{6.14}
$$

这里要求：

$$
p_t(x)>0
$$

---

## 6.4 使用贝叶斯公式得到条件期望

根据贝叶斯公式：

$$
\boxed{
p_t(y\mid x)
=
\frac{p_t(x\mid y)q(y)}{p_t(x)}
}
\tag{6.15}
$$

**将式 (6.15) 代入式 (6.14)：**

$$
u_t(x)
=
\int
u_t(x\mid y)p_t(y\mid x)\,dy
\tag{6.16}
$$

根据条件期望的定义：

$$
\boxed{
u_t(x)
=
\mathbb E[
u_t(x\mid Y)\mid X_t=x
]
}
\tag{6.17}
$$

这就是论文第一个核心定理。

### 理论含义

条件速度场：

$$
u_t(x\mid y)
$$

描述已知目标数据 $y$ 时，条件粒子群应该如何运动。

边缘速度场：

$$
u_t(x)
$$

描述不知道具体目标数据、只知道当前位置 $x$ 时，整个粒子群应该如何运动。

它等于所有可能条件速度按照后验概率的加权平均。

### 为什么式 (6.13) 中出现条件速度？

式 (6.13) 不是直接通过全概率公式得到的。

其推导顺序是：

$$
\boxed{
\begin{aligned}
&\text{条件连续性方程}\\
&\Downarrow\\
&\text{对真实数据 }y\text{ 积分}\\
&\Downarrow\\
&\text{构造边缘概率流量}\\
&\Downarrow\\
&\text{定义边缘速度}\\
&\Downarrow\\
&\text{贝叶斯后验平均}
\end{aligned}
}
$$

因此条件速度首先作为条件概率流量的一部分出现。

**并不是在边缘概率公式中随意乘上一个速度。**

---

# 第七章：Conditional Flow Matching

## 7.1 FM 损失

原始 Flow Matching 损失：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E_{t,X_t}
\|v_\theta(X_t,t)-u_t(X_t)\|^2
}
\tag{7.1}
$$

其中：

$$
X_t\sim p_t
$$

训练目标为边缘速度：

$$
u_t(X_t)
$$

但它需要计算式 (6.14) 中的积分。

---

## 7.2 CFM 损失

Conditional Flow Matching 改为：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}
=
\mathbb E_{t,Y,X_t\mid Y}
\|v_\theta(X_t,t)-u_t(X_t\mid Y)\|^2
}
\tag{7.2}
$$

其中：

$$
Y\sim q
$$

$$
X_t\sim p_t(\cdot\mid Y)
$$

网络输入仍然是：

$$
(X_t,t)
$$

监督目标变成：

$$
u_t(X_t\mid Y)
$$

这里的 Conditional 指训练监督使用条件概率路径对应的条件速度场。

它并不表示网络必须输入类别、文本或者自动驾驶场景编码。

---

## 7.3 MSE 最优解是条件期望

考虑一般随机变量：

$$
X,U
$$

定义：

$$
\mathcal L(f)
=
\mathbb E\|f(X)-U\|^2
\tag{7.3}
$$

定义：

$$
m(X)=\mathbb E[U\mid X]
\tag{7.4}
$$

将误差写成：

$$
f(X)-U
=
[f(X)-m(X)]+[m(X)-U]
\tag{7.5}
$$

对式 (7.5) 两侧平方：

$$
\begin{aligned}
\|f(X)-U\|^2
={}&
\|f(X)-m(X)\|^2\\
&+\|m(X)-U\|^2\\
&+2[f(X)-m(X)]^\top[m(X)-U]
\end{aligned}
\tag{7.6}
$$

对式 (7.6) 取期望。

由于：

$$
\mathbb E[m(X)-U\mid X]=0
\tag{7.7}
$$

因此：

$$
\mathbb E[
(f(X)-m(X))^\top(m(X)-U)
]=0
\tag{7.8}
$$

**将式 (7.8) 代入式 (7.6) 的期望表达式：**

$$
\boxed{
\mathcal L(f)
=
\mathbb E\|f(X)-m(X)\|^2
+
\mathbb E\|m(X)-U\|^2
}
\tag{7.9}
$$

第二项与函数 $f$ 无关。

因此最优解：

$$
\boxed{
f^*(x)=\mathbb E[U\mid X=x]
}
\tag{7.10}
$$

---

## 7.4 CFM 的最优解

在 CFM 中，令：

$$
X=(X_t,t)
\tag{7.11}
$$

$$
U=u_t(X_t\mid Y)
\tag{7.12}
$$

根据式 (7.10)：

$$
v_\theta^*(x,t)
=
\mathbb E[
u_t(x\mid Y)\mid X_t=x,t
]
\tag{7.13}
$$

根据式 (6.17)：

$$
u_t(x)
=
\mathbb E[
u_t(x\mid Y)\mid X_t=x
]
$$

这里将 $t$ 视为固定时间参数。

因此：

$$
\boxed{
v_\theta^*(x,t)=u_t(x)
}
\tag{7.14}
$$

这说明 CFM 虽然使用条件速度监督，但在总体最优情况下，网络学习到的是边缘速度场。

---

## 7.5 CFM 与 FM 的梯度等价证明

定义：

$$
V=v_\theta(X_t,t)
\tag{7.15}
$$

$$
U=u_t(X_t\mid Y)
\tag{7.16}
$$

$$
M=u_t(X_t)
\tag{7.17}
$$

根据式 (6.17)：

$$
M=\mathbb E[U\mid X_t,t]
\tag{7.18}
$$

### 第一步：展开 CFM 损失

根据式 (7.2)：

$$
\mathcal L_{\mathrm{CFM}}
=
\mathbb E\|V-U\|^2
\tag{7.19}
$$

平方展开：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}
=
\mathbb E\|V\|^2
-
2\mathbb E[V^\top U]
+
\mathbb E\|U\|^2
}
\tag{7.20}
$$

### 第二步：展开 FM 损失

根据式 (7.1)：

$$
\mathcal L_{\mathrm{FM}}
=
\mathbb E\|V-M\|^2
\tag{7.21}
$$

平方展开：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E\|V\|^2
-
2\mathbb E[V^\top M]
+
\mathbb E\|M\|^2
}
\tag{7.22}
$$

### 第三步：证明交叉项相等

考虑：

$$
\mathbb E[V^\top U]
\tag{7.23}
$$

使用全期望公式：

$$
\mathbb E[V^\top U]
=
\mathbb E[
\mathbb E[V^\top U\mid X_t,t]
]
\tag{7.24}
$$

由于：

$$
V=v_\theta(X_t,t)
$$

在固定 $X_t,t$ 后是确定值，因此：

$$
\mathbb E[V^\top U\mid X_t,t]
=
V^\top\mathbb E[U\mid X_t,t]
\tag{7.25}
$$

根据式 (7.18)：

$$
\mathbb E[U\mid X_t,t]=M
$$

**将式 (7.18) 代入式 (7.25)：**

$$
\mathbb E[V^\top U\mid X_t,t]
=
V^\top M
\tag{7.26}
$$

**将式 (7.26) 代入式 (7.24)：**

$$
\boxed{
\mathbb E[V^\top U]
=
\mathbb E[V^\top M]
}
\tag{7.27}
$$

### 第四步：比较损失

将式 (7.20) 减去式 (7.22)：

$$
\begin{aligned}
\mathcal L_{\mathrm{CFM}}
-
\mathcal L_{\mathrm{FM}}
={}&
-2\mathbb E[V^\top U]
+2\mathbb E[V^\top M]\\
&+\mathbb E\|U\|^2
-\mathbb E\|M\|^2
\end{aligned}
\tag{7.28}
$$

**将式 (7.27) 代入式 (7.28)：**

$$
\boxed{
\mathcal L_{\mathrm{CFM}}
-
\mathcal L_{\mathrm{FM}}
=
\mathbb E\|U\|^2
-
\mathbb E\|M\|^2
}
\tag{7.29}
$$

右侧与神经网络参数：

$$
\theta
$$

无关。

因此：

$$
\boxed{
\nabla_\theta\mathcal L_{\mathrm{CFM}}
=
\nabla_\theta\mathcal L_{\mathrm{FM}}
}
\tag{7.30}
$$

这就是论文第二个核心定理。

### 进一步理解

根据条件方差分解：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}
=
\mathcal L_{\mathrm{FM}}
+
\mathbb E
\|U-\mathbb E[U\mid X_t,t]\|^2
}
\tag{7.31}
$$

第二项是条件速度监督中无法由当前位置完全解释的方差。

它与网络参数无关。

因此：

**CFM 和 FM 的损失数值不一定相同，但其总体梯度相同。**

---

# 第八章：高斯条件概率路径

## 8.1 高斯条件路径定义

论文选择：

$$
\boxed{
p_t(x\mid y)
=
\mathcal N(x;\mu_t(y),\sigma_t(y)^2I)
}
\tag{8.1}
$$

其中：

- $\mu_t(y)$：条件均值。
- $\sigma_t(y)>0$：条件标准差。
- $y$：固定的真实数据样本。

定义：

$$
Z\sim\mathcal N(0,I)
\tag{8.2}
$$

构造仿射映射：

$$
\boxed{
\psi_t(z;y)
=
\sigma_t(y)z+\mu_t(y)
}
\tag{8.3}
$$

令：

$$
X_t=\psi_t(Z;y)
\tag{8.4}
$$

由于高斯随机变量经过仿射变换仍然是高斯分布：

$$
\boxed{
X_t\mid Y=y
\sim
\mathcal N(\mu_t(y),\sigma_t(y)^2I)
}
\tag{8.5}
$$

因此式 (8.3) 可以生成式 (8.1) 对应的条件概率路径。

---

## 8.2 仿射变换作用于哪个随机变量？

固定：

$$
Y=y
$$

之后：

$$
\mu_t(y),\sigma_t(y)
$$

都是确定的时间函数。

此时唯一的随机变量是：

$$
Z
$$

所以：

$$
X_t=\sigma_t(y)Z+\mu_t(y)
$$

是对随机变量 $Z$ 的仿射变换。

如果不固定 $Y$，则：

$$
X_t=\sigma_t(Y)Z+\mu_t(Y)
\tag{8.6}
$$

同时包含：

$$
Y,Z
$$

的随机性。

因此：

$$
p_t(x\mid y)
$$

是条件概率密度。

而：

$$
p_t(x)
=
\int p_t(x\mid y)q(y)\,dy
$$

是边缘概率密度。

---

## 8.3 高斯条件速度场推导

根据式 (8.3)：

$$
\psi_t(z;y)
=
\sigma_t(y)z+\mu_t(y)
$$

固定 $z,y$，对时间求导：

$$
\boxed{
\frac{d}{dt}\psi_t(z;y)
=
\dot\sigma_t(y)z+\dot\mu_t(y)
}
\tag{8.7}
$$

其中：

$$
\dot\sigma_t(y)
=
\frac{\partial\sigma_t(y)}{\partial t}
$$

$$
\dot\mu_t(y)
=
\frac{\partial\mu_t(y)}{\partial t}
$$

根据式 (8.3)：

$$
x=\sigma_t(y)z+\mu_t(y)
\tag{8.8}
$$

移项：

$$
x-\mu_t(y)=\sigma_t(y)z
$$

两侧除以：

$$
\sigma_t(y)
$$

得到：

$$
\boxed{
z=
\frac{x-\mu_t(y)}{\sigma_t(y)}
}
\tag{8.9}
$$

**将式 (8.9) 代入式 (8.7)：**

$$
\frac{d}{dt}\psi_t(z;y)
=
\dot\sigma_t(y)
\frac{x-\mu_t(y)}{\sigma_t(y)}
+
\dot\mu_t(y)
\tag{8.10}
$$

定义条件速度场：

$$
u_t(x\mid y)
=
\left.
\frac{d}{dt}\psi_t(z;y)
\right|_{z=\psi_t^{-1}(x;y)}
\tag{8.11}
$$

因此：

$$
\boxed{
u_t(x\mid y)
=
\frac{\dot\sigma_t(y)}{\sigma_t(y)}
[x-\mu_t(y)]
+
\dot\mu_t(y)
}
\tag{8.12}
$$

这是论文第三个核心定理。

### 为什么它是条件速度而不是边缘速度？

因为整个推导始终固定：

$$
Y=y
$$

得到的是固定条件概率路径：

$$
p_t(x\mid y)
$$

对应的速度场。

边缘速度还需要进一步计算：

$$
\boxed{
u_t(x)
=
\mathbb E[
u_t(x\mid Y)\mid X_t=x
]
}
\tag{8.13}
$$

---

## 8.4 高斯 CFM 的训练公式

根据式 (7.2)：

$$
\mathcal L_{\mathrm{CFM}}
=
\mathbb E
\|v_\theta(X_t,t)-u_t(X_t\mid Y)\|^2
\tag{8.14}
$$

根据式 (8.6)：

$$
X_t=\sigma_t(Y)Z+\mu_t(Y)
$$

根据式 (8.7)：

$$
u_t(X_t\mid Y)
=
\dot\sigma_t(Y)Z+\dot\mu_t(Y)
\tag{8.15}
$$

**将式 (8.6) 和式 (8.15) 代入式 (8.14)：**

$$
\boxed{
\begin{aligned}
\mathcal L_{\mathrm{CFM}}
=
\mathbb E_{t,Y,Z}
\big\|
&v_\theta(
\sigma_t(Y)Z+\mu_t(Y),t
)\\
&-
[\dot\sigma_t(Y)Z+\dot\mu_t(Y)]
\big\|^2
\end{aligned}
}
\tag{8.16}
$$

这就是高斯条件路径下的 CFM 训练公式。

训练时只需要：

1. 采样真实数据 $Y$。
2. 采样高斯噪声 $Z$。
3. 采样时间 $t$。
4. 计算中间状态 $X_t$。
5. 计算条件速度标签。
6. 使用 MSE 训练速度网络。

不需要显式计算边缘概率密度或边缘速度。

---

# 第九章：Optimal Transport Probability Path

## 9.1 OT 条件路径定义

论文选择：

$$
\boxed{
\mu_t(y)=ty
}
\tag{9.1}
$$

$$
\boxed{
\sigma_t=1-(1-\sigma_{\min})t
}
\tag{9.2}
$$

其中：

$$
0<\sigma_{\min}<1
$$

将式 (9.1) 和式 (9.2) 代入高斯条件路径式 (8.1)：

$$
\boxed{
p_t(x\mid y)
=
\mathcal N
\left(
x;
ty,
[1-(1-\sigma_{\min})t]^2I
\right)
}
\tag{9.3}
$$

初始：

$$
t=0
$$

得到：

$$
p_0(x\mid y)=\mathcal N(0,I)
\tag{9.4}
$$

终点：

$$
t=1
$$

得到：

$$
p_1(x\mid y)
=
\mathcal N(y,\sigma_{\min}^2I)
\tag{9.5}
$$

当：

$$
\sigma_{\min}\rightarrow0
$$

时，终点条件分布趋近集中在 $y$ 的点质量分布。

---

## 9.2 OT 条件速度场推导

根据式 (8.12)：

$$
u_t(x\mid y)
=
\frac{\dot\sigma_t}{\sigma_t}
(x-\mu_t)
+
\dot\mu_t
\tag{9.6}
$$

根据式 (9.1)：

$$
\dot\mu_t(y)=y
\tag{9.7}
$$

根据式 (9.2)：

$$
\dot\sigma_t=-(1-\sigma_{\min})
\tag{9.8}
$$

**将式 (9.1)、式 (9.2)、式 (9.7)、式 (9.8) 代入式 (9.6)：**

$$
u_t(x\mid y)
=
-\frac{1-\sigma_{\min}}
{1-(1-\sigma_{\min})t}
(x-ty)
+
y
\tag{9.9}
$$

通分：

$$
u_t(x\mid y)
=
\frac{
-(1-\sigma_{\min})(x-ty)
+
y[1-(1-\sigma_{\min})t]
}{
1-(1-\sigma_{\min})t
}
\tag{9.10}
$$

展开分子：

$$
\begin{aligned}
&-(1-\sigma_{\min})x
+(1-\sigma_{\min})ty\\
&\quad+y-(1-\sigma_{\min})ty
\end{aligned}
$$

中间两项抵消，得到：

$$
\boxed{
u_t(x\mid y)
=
\frac{
y-(1-\sigma_{\min})x
}{
1-(1-\sigma_{\min})t
}
}
\tag{9.11}
$$

---

## 9.3 OT 条件流映射

根据高斯仿射映射式 (8.3)：

$$
\psi_t(z;y)=\sigma_tz+\mu_t(y)
$$

**将式 (9.1) 和式 (9.2) 代入：**

$$
\boxed{
\psi_t(z;y)
=
[1-(1-\sigma_{\min})t]z+ty
}
\tag{9.12}
$$

对式 (9.12) 关于时间求导：

$$
\frac{d}{dt}\psi_t(z;y)
=
-(1-\sigma_{\min})z+y
\tag{9.13}
$$

因此：

$$
\boxed{
\frac{d}{dt}\psi_t(z;y)
=
y-(1-\sigma_{\min})z
}
\tag{9.14}
$$

由于 $z,y$ 固定，右侧不依赖时间。

再次对时间求导：

$$
\boxed{
\frac{d^2}{dt^2}\psi_t(z;y)=0
}
\tag{9.15}
$$

说明条件粒子轨迹是匀速直线。

---

## 9.4 为什么称为 Optimal Transport？

考虑两个高斯分布：

$$
p_0=\mathcal N(0,I)
\tag{9.16}
$$

$$
p_1(\cdot\mid y)
=
\mathcal N(y,\sigma_{\min}^2I)
\tag{9.17}
$$

在平方欧氏距离成本下，它们之间的最优传输映射为：

$$
\boxed{
T_y(z)=y+\sigma_{\min}z
}
\tag{9.18}
$$

使用位移插值：

$$
\psi_t(z;y)
=
(1-t)z+tT_y(z)
\tag{9.19}
$$

**将式 (9.18) 代入式 (9.19)：**

$$
\psi_t(z;y)
=
(1-t)z+t(y+\sigma_{\min}z)
\tag{9.20}
$$

展开：

$$
=
(1-t)z+ty+t\sigma_{\min}z
$$

整理：

$$
\boxed{
\psi_t(z;y)
=
[1-(1-\sigma_{\min})t]z+ty
}
\tag{9.21}
$$

式 (9.21) 与式 (9.12) 完全相同。

因此，这条条件路径是两个条件高斯分布之间的 OT 位移插值。

### 重要区别

$$
\boxed{
\text{条件高斯之间的 OT}
\neq
\text{初始高斯到完整真实分布的全局 OT}
}
\tag{9.22}
$$

因此：

**条件路径是直线，不代表最终边缘 ODE 的轨迹也是直线。**

---

## 9.5 OT-CFM 的最终训练公式

根据式 (9.12)：

$$
X_t
=
[1-(1-\sigma_{\min})t]Z+tY
\tag{9.23}
$$

根据式 (9.14)：

$$
U=Y-(1-\sigma_{\min})Z
\tag{9.24}
$$

将式 (9.23) 和式 (9.24) 代入 CFM 损失式 (7.2)：

$$
\boxed{
\begin{aligned}
\mathcal L_{\mathrm{OT-CFM}}
=
\mathbb E_{t,Y,Z}
\big\|
&v_\theta(
[1-(1-\sigma_{\min})t]Z+tY,t
)\\
&-
[Y-(1-\sigma_{\min})Z]
\big\|^2
\end{aligned}
}
\tag{9.25}
$$

当：

$$
\sigma_{\min}\rightarrow0
$$

得到：

$$
\boxed{
X_t=(1-t)Z+tY
}
\tag{9.26}
$$

$$
\boxed{
U=Y-Z
}
\tag{9.27}
$$

因此：

$$
\boxed{
\mathcal L
=
\mathbb E_{t,Y,Z}
\left\|
v_\theta((1-t)Z+tY,t)-(Y-Z)
\right\|^2
}
\tag{9.28}
$$

这就是常用的线性 Flow Matching 训练公式。

---

# 第十章：为什么条件路径是直线，边缘 ODE 轨迹却可能弯曲？

## 10.1 条件路径的速度

考虑：

$$
X_t=(1-t)Z+tY
\tag{10.1}
$$

固定：

$$
Z=z,\qquad Y=y
$$

得到：

$$
x_t=(1-t)z+ty
\tag{10.2}
$$

对时间求导：

$$
\boxed{
\dot x_t=y-z
}
\tag{10.3}
$$

再次求导：

$$
\boxed{
\ddot x_t=0
}
\tag{10.4}
$$

所以条件粒子轨迹是匀速直线。

---

## 10.2 边缘速度的数学表达

根据式 (6.17)：

$$
u_t(x)
=
\mathbb E[Y-Z\mid X_t=x]
\tag{10.5}
$$

根据式 (10.1)：

$$
X_t=(1-t)Z+tY
$$

整理：

$$
Z=\frac{X_t-tY}{1-t}
\tag{10.6}
$$

因此：

$$
Y-Z
=
Y-\frac{X_t-tY}{1-t}
\tag{10.7}
$$

通分：

$$
Y-Z
=
\frac{(1-t)Y-X_t+tY}{1-t}
$$

整理：

$$
\boxed{
Y-Z=\frac{Y-X_t}{1-t}
}
\tag{10.8}
$$

**将式 (10.8) 代入式 (10.5)：**

$$
u_t(x)
=
\mathbb E
\left[
\frac{Y-X_t}{1-t}
\mid X_t=x
\right]
\tag{10.9}
$$

由于：

$$
X_t=x
$$

是已知条件，因此：

$$
\boxed{
u_t(x)
=
\frac{
\mathbb E[Y\mid X_t=x]-x
}{
1-t
}
}
\tag{10.10}
$$

其中：

$$
\mathbb E[Y\mid X_t=x]
$$

通常依赖当前位置 $x$ 和时间 $t$。

因此边缘速度场一般不是恒定速度场。

---

## 10.3 边缘 ODE 的加速度

边缘 ODE：

$$
\boxed{
\dot X_t=u_t(X_t)
}
\tag{10.11}
$$

对式 (10.11) 关于时间求导：

$$
\ddot X_t
=
\frac{d}{dt}u_t(X_t)
\tag{10.12}
$$

根据链式法则：

$$
\ddot X_t
=
\partial_tu_t(X_t)
+
J_{u_t}(X_t)\dot X_t
\tag{10.13}
$$

其中：

$$
J_{u_t}(x)
=
\frac{\partial u_t(x)}{\partial x}
\tag{10.14}
$$

**将式 (10.11) 代入式 (10.13)：**

$$
\boxed{
\ddot X_t
=
\partial_tu_t(X_t)
+
J_{u_t}(X_t)u_t(X_t)
}
\tag{10.15}
$$

因此，即使条件速度恒定，边缘 ODE 的加速度仍然可能不为零。

需要进一步区分：

- 加速度不为零，说明速度发生变化。
- 加速度具有垂直于速度的分量，才意味着轨迹方向发生变化。
- 因此速度不恒定并不必然意味着几何轨迹弯曲。

---

## 10.4 二维高斯例子：边缘轨迹确实可以弯曲

设：

$$
Z\sim\mathcal N(0,I_2)
\tag{10.16}
$$

$$
Y\sim\mathcal N(0,\Sigma)
\tag{10.17}
$$

其中：

$$
\Sigma=
\begin{bmatrix}
4&0\\
0&1
\end{bmatrix}
\tag{10.18}
$$

假设 $Z,Y$ 独立。

定义：

$$
X_t=(1-t)Z+tY
\tag{10.19}
$$

### 第一步：计算边缘分布

由于独立高斯随机变量的线性组合仍然是高斯分布：

$$
X_t\sim\mathcal N(0,C_t)
\tag{10.20}
$$

其中：

$$
C_t=(1-t)^2I+t^2\Sigma
\tag{10.21}
$$

**将式 (10.18) 代入式 (10.21)：**

$$
C_t=
\begin{bmatrix}
(1-t)^2+4t^2&0\\
0&(1-t)^2+t^2
\end{bmatrix}
\tag{10.22}
$$

定义：

$$
c_1(t)=(1-t)^2+4t^2
\tag{10.23}
$$

$$
c_2(t)=(1-t)^2+t^2
\tag{10.24}
$$

### 第二步：计算条件期望

根据式 (10.10)：

$$
u_t(x)
=
\frac{\mathbb E[Y\mid X_t=x]-x}{1-t}
\tag{10.25}
$$

由于 $Y,X_t$ 联合高斯：

$$
\mathbb E[Y\mid X_t=x]
=
\operatorname{Cov}(Y,X_t)
C_t^{-1}x
\tag{10.26}
$$

根据式 (10.19)：

$$
X_t=(1-t)Z+tY
$$

计算协方差：

$$
\begin{aligned}
\operatorname{Cov}(Y,X_t)
={}&
(1-t)\operatorname{Cov}(Y,Z)\\
&+t\operatorname{Cov}(Y,Y)
\end{aligned}
\tag{10.27}
$$

由于 $Y,Z$ 独立：

$$
\operatorname{Cov}(Y,Z)=0
$$

且：

$$
\operatorname{Cov}(Y,Y)=\Sigma
$$

所以：

$$
\boxed{
\operatorname{Cov}(Y,X_t)=t\Sigma
}
\tag{10.28}
$$

**将式 (10.28) 代入式 (10.26)：**

$$
\boxed{
\mathbb E[Y\mid X_t=x]
=
t\Sigma C_t^{-1}x
}
\tag{10.29}
$$

### 第三步：计算边缘速度

**将式 (10.29) 代入式 (10.25)：**

$$
u_t(x)
=
\frac{t\Sigma C_t^{-1}x-x}{1-t}
\tag{10.30}
$$

由于：

$$
x=C_tC_t^{-1}x
$$

所以：

$$
u_t(x)
=
\frac{t\Sigma-C_t}{1-t}C_t^{-1}x
\tag{10.31}
$$

根据式 (10.21)：

$$
C_t=(1-t)^2I+t^2\Sigma
$$

计算：

$$
\begin{aligned}
t\Sigma-C_t
&=t\Sigma-(1-t)^2I-t^2\Sigma\\
&=t(1-t)\Sigma-(1-t)^2I\\
&=(1-t)[t\Sigma-(1-t)I]
\end{aligned}
\tag{10.32}
$$

**将式 (10.32) 代入式 (10.31)：**

$$
\boxed{
u_t(x)
=
[t\Sigma-(1-t)I]C_t^{-1}x
}
\tag{10.33}
$$

根据式 (10.18) 和式 (10.22)：

$$
\boxed{
u_t(x)=
\begin{bmatrix}
\displaystyle\frac{5t-1}{c_1(t)}&0\\[7pt]
0&\displaystyle\frac{2t-1}{c_2(t)}
\end{bmatrix}x
}
\tag{10.34}
$$

这是解析计算得到的真实边缘速度场。

### 第四步：求解边缘 ODE

根据：

$$
\dot X_t=u_t(X_t)
$$

以及式 (10.34)：

$$
\dot x_{t,1}
=
\frac{5t-1}{c_1(t)}x_{t,1}
\tag{10.35}
$$

$$
\dot x_{t,2}
=
\frac{2t-1}{c_2(t)}x_{t,2}
\tag{10.36}
$$

对式 (10.23) 求导：

$$
c_1'(t)=2(5t-1)
\tag{10.37}
$$

对式 (10.24) 求导：

$$
c_2'(t)=2(2t-1)
\tag{10.38}
$$

**将式 (10.37) 代入式 (10.35)：**

$$
\frac{\dot x_{t,1}}{x_{t,1}}
=
\frac12\frac{c_1'(t)}{c_1(t)}
\tag{10.39}
$$

两侧对时间积分：

$$
\log|x_{t,1}|
=
\frac12\log c_1(t)+C
\tag{10.40}
$$

结合初始条件：

$$
x_{0,1}
$$

得到：

$$
x_{t,1}
=
\sqrt{\frac{c_1(t)}{c_1(0)}}x_{0,1}
\tag{10.41}
$$

同理，根据式 (10.38) 和式 (10.36)：

$$
x_{t,2}
=
\sqrt{\frac{c_2(t)}{c_2(0)}}x_{0,2}
\tag{10.42}
$$

由于：

$$
c_1(0)=c_2(0)=1
$$

因此：

$$
\boxed{
X_t=
\begin{bmatrix}
\sqrt{c_1(t)}X_{0,1}\\
\sqrt{c_2(t)}X_{0,2}
\end{bmatrix}
}
\tag{10.43}
$$

### 第五步：验证轨迹不是直线

取初始位置：

$$
X_0=(1,1)
\tag{10.44}
$$

根据式 (10.43)：

$$
X_t=
\left(
\sqrt{(1-t)^2+4t^2},
\sqrt{(1-t)^2+t^2}
\right)
\tag{10.45}
$$

当：

$$
t=0
$$

有：

$$
X_0=(1,1)
\tag{10.46}
$$

当：

$$
t=1
$$

有：

$$
X_1=(2,1)
\tag{10.47}
$$

当：

$$
t=0.5
$$

有：

$$
X_{0.5}
=
(\sqrt{1.25},\sqrt{0.5})
\tag{10.48}
$$

即：

$$
\boxed{
X_{0.5}\approx(1.118,0.707)
}
\tag{10.49}
$$

连接端点：

$$
(1,1)\rightarrow(2,1)
$$

的直线满足：

$$
x_2=1
$$

但中间点：

$$
x_2\approx0.707
$$

因此边缘 ODE 的样本轨迹不是直线。

**结论：**

$$
\boxed{
\text{条件粒子轨迹是直线}
\not\Rightarrow
\text{边缘 ODE 粒子轨迹是直线}
}
\tag{10.50}
$$

即使神经网络完全准确，边缘 ODE 的轨迹也可能弯曲。

这不是模型预测误差，而是边缘化后真实动力学的性质。

---

# 第十一章：Diffusion Probability Path

## 11.1 扩散模型的条件概率路径

扩散模型通常使用：

$$
\boxed{
X_\tau=\alpha_\tau Y+\sigma_\tau Z
}
\tag{11.1}
$$

其中：

- $\tau$：前向扩散时间，从数据到噪声。
- $Y$：真实数据。
- $Z\sim\mathcal N(0,I)$：高斯噪声。
- $\alpha_\tau$：数据保留系数。
- $\sigma_\tau$：噪声标准差。

固定：

$$
Y=y
$$

得到：

$$
\boxed{
p_\tau(x\mid y)
=
\mathcal N(x;\alpha_\tau y,\sigma_\tau^2I)
}
\tag{11.2}
$$

这属于第八章介绍的高斯条件概率路径。

因此，扩散模型中的 VP、VE 等概率路径可以放入 Flow Matching 框架研究。

需要注意：这里统一的是**概率路径及其对应的确定性速度场**，并不意味着 SDE 与 ODE 是完全相同的随机过程。

---

## 11.2 VP-SDE

前向 VP-SDE：

$$
\boxed{
dX_\tau
=
-\frac12\beta(\tau)X_\tau d\tau
+
\sqrt{\beta(\tau)}dW_\tau
}
\tag{11.3}
$$

其中：

- $\beta(\tau)>0$：噪声调度函数。
- $W_\tau$：标准布朗运动。

其条件边缘分布可以表示为：

$$
X_\tau
=
\alpha_\tau Y+
\sqrt{1-\alpha_\tau^2}Z
\tag{11.4}
$$

其中：

$$
\boxed{
\alpha_\tau
=
\exp
\left(
-\frac12\int_0^\tau\beta(s)\,ds
\right)
}
\tag{11.5}
$$

为了使用从噪声到数据的生成时间，定义：

$$
t=1-\tau
\tag{11.6}
$$

令：

$$
a(t)=\alpha_{1-t}
\tag{11.7}
$$

则：

$$
\mu_t(y)=a(t)y
\tag{11.8}
$$

$$
\sigma_t=\sqrt{1-a(t)^2}
\tag{11.9}
$$

根据高斯条件速度公式 (8.12)：

$$
u_t(x\mid y)
=
\frac{\dot\sigma_t}{\sigma_t}
(x-\mu_t)
+
\dot\mu_t
\tag{11.10}
$$

计算：

$$
\dot\mu_t(y)=\dot a(t)y
\tag{11.11}
$$

对式 (11.9) 求导：

$$
\dot\sigma_t
=
\frac12(1-a^2)^{-1/2}(-2a\dot a)
$$

因此：

$$
\boxed{
\dot\sigma_t
=
-\frac{a\dot a}{\sqrt{1-a^2}}
}
\tag{11.12}
$$

根据式 (11.9)：

$$
\boxed{
\frac{\dot\sigma_t}{\sigma_t}
=
-\frac{a\dot a}{1-a^2}
}
\tag{11.13}
$$

**将式 (11.8)、式 (11.11)、式 (11.13) 代入式 (11.10)：**

$$
u_t(x\mid y)
=
-\frac{a\dot a}{1-a^2}(x-ay)
+
\dot a y
\tag{11.14}
$$

展开：

$$
u_t(x\mid y)
=
-\frac{a\dot a}{1-a^2}x
+
\frac{a^2\dot a}{1-a^2}y
+
\dot a y
\tag{11.15}
$$

将后两项合并：

$$
\frac{a^2\dot a}{1-a^2}y+\dot a y
=
\frac{\dot a}{1-a^2}y
$$

因此：

$$
\boxed{
u_t(x\mid y)
=
\frac{\dot a(t)}{1-a(t)^2}
[y-a(t)x]
}
\tag{11.16}
$$

这就是 VP 类条件高斯路径对应的速度场。

---

## 11.3 VE-SDE

VE 类扩散过程的条件边缘通常表示为：

$$
\boxed{
X_\tau=Y+\sigma(\tau)Z
}
\tag{11.17}
$$

固定：

$$
Y=y
$$

有：

$$
p_\tau(x\mid y)
=
\mathcal N(x;y,\sigma(\tau)^2I)
\tag{11.18}
$$

转换到生成时间：

$$
t=1-\tau
$$

定义：

$$
\mu_t(y)=y
\tag{11.19}
$$

$$
\sigma_t=\sigma(1-t)
\tag{11.20}
$$

因此：

$$
\dot\mu_t=0
\tag{11.21}
$$

$$
\dot\sigma_t=-\sigma'(1-t)
\tag{11.22}
$$

**将式 (11.19) 至式 (11.22) 代入式 (8.12)：**

$$
\boxed{
u_t(x\mid y)
=
-\frac{\sigma'(1-t)}{\sigma(1-t)}
(x-y)
}
\tag{11.23}
$$

因此 VE 类条件概率路径同样可以构造解析速度监督。

---

# 第十二章：Probability Flow ODE 与 SDE 的关系

## 12.1 一般 SDE

考虑：

$$
\boxed{
dX_\tau
=
f_\tau(X_\tau)d\tau
+
g_\tau dW_\tau
}
\tag{12.1}
$$

其中：

- $f_\tau(x)$：漂移项。
- $g_\tau$：扩散系数，这里假设不依赖空间位置。
- $W_\tau$：标准布朗运动。

对应的概率密度：

$$
p_\tau(x)
$$

满足 Fokker–Planck 方程：

$$
\boxed{
\partial_\tau p_\tau
=
-\nabla\cdot(f_\tau p_\tau)
+
\frac12g_\tau^2\Delta p_\tau
}
\tag{12.2}
$$

---

## 12.2 引入 Score

定义：

$$
\boxed{
s_\tau(x)=\nabla_x\log p_\tau(x)
}
\tag{12.3}
$$

根据对数求导：

$$
\nabla\log p_\tau
=
\frac{\nabla p_\tau}{p_\tau}
\tag{12.4}
$$

两侧乘以：

$$
p_\tau
$$

得到：

$$
\boxed{
\nabla p_\tau=p_\tau s_\tau
}
\tag{12.5}
$$

对式 (12.5) 两侧取散度：

$$
\nabla\cdot(\nabla p_\tau)
=
\nabla\cdot(p_\tau s_\tau)
$$

因此：

$$
\boxed{
\Delta p_\tau
=
\nabla\cdot(p_\tau s_\tau)
}
\tag{12.6}
$$

---

## 12.3 推导 Probability Flow ODE

根据 Fokker–Planck 方程 (12.2)：

$$
\partial_\tau p_\tau
=
-\nabla\cdot(f_\tau p_\tau)
+
\frac12g_\tau^2\Delta p_\tau
$$

**将式 (12.6) 代入式 (12.2)：**

$$
\partial_\tau p_\tau
=
-\nabla\cdot(f_\tau p_\tau)
+
\frac12g_\tau^2\nabla\cdot(p_\tau s_\tau)
\tag{12.7}
$$

由于 $g_\tau$ 不依赖 $x$，可以合并散度：

$$
\partial_\tau p_\tau
=
-\nabla\cdot
\left[
p_\tau
\left(
f_\tau-\frac12g_\tau^2s_\tau
\right)
\right]
\tag{12.8}
$$

定义：

$$
\boxed{
u_\tau(x)
=
f_\tau(x)
-
\frac12g_\tau^2s_\tau(x)
}
\tag{12.9}
$$

**将式 (12.9) 代入式 (12.8)：**

$$
\boxed{
\partial_\tau p_\tau
+
\nabla\cdot(p_\tau u_\tau)
=0
}
\tag{12.10}
$$

式 (12.10) 是标准连续性方程。

因此确定性 ODE：

$$
\boxed{
\frac{dX_\tau}{d\tau}
=
f_\tau(X_\tau)
-
\frac12g_\tau^2s_\tau(X_\tau)
}
\tag{12.11}
$$

能够生成与式 (12.1) 相同的单时刻边缘概率分布。

这就是 Probability Flow ODE。

### 关键结论

$$
\boxed{
\text{SDE 与 Probability Flow ODE 可以具有相同边缘分布}
}
\tag{12.12}
$$

但：

$$
\boxed{
\text{相同边缘分布}
\neq
\text{相同粒子轨迹}
}
\tag{12.13}
$$

SDE 的样本轨迹具有随机性。

ODE 在固定初始状态后，其样本轨迹由确定性动力学决定。

此外，式 (12.11) 使用前向扩散时间 $\tau$。生成时需要从噪声端向数据端反向积分，或者转换到生成时间 $t=1-\tau$。

---

# 第十三章：Flow Matching 与 Score Matching

## 13.1 条件 Score

考虑高斯加噪路径：

$$
X_t=\alpha_tY+\sigma_tZ
\tag{13.1}
$$

固定：

$$
Y=y
$$

得到：

$$
p_t(x\mid y)
=
\mathcal N(x;\alpha_ty,\sigma_t^2I)
\tag{13.2}
$$

定义条件 Score：

$$
s_t(x\mid y)
=
\nabla_x\log p_t(x\mid y)
\tag{13.3}
$$

高斯分布的对数密度：

$$
\log p_t(x\mid y)
=
-\frac{\|x-\alpha_ty\|^2}{2\sigma_t^2}
+C
\tag{13.4}
$$

其中 $C$ 与 $x$ 无关。

对式 (13.4) 关于 $x$ 求梯度：

$$
\boxed{
s_t(x\mid y)
=
-\frac{x-\alpha_ty}{\sigma_t^2}
}
\tag{13.5}
$$

根据式 (13.1)：

$$
x-\alpha_ty=\sigma_tz
\tag{13.6}
$$

**将式 (13.6) 代入式 (13.5)：**

$$
\boxed{
s_t(x\mid y)
=
-\frac z{\sigma_t}
}
\tag{13.7}
$$

---

## 13.2 边缘 Score 的推导

根据全概率公式：

$$
p_t(x)
=
\int p_t(x\mid y)q(y)\,dy
\tag{13.8}
$$

对式 (13.8) 关于 $x$ 求梯度：

$$
\nabla p_t(x)
=
\int\nabla p_t(x\mid y)q(y)\,dy
\tag{13.9}
$$

根据 Score 定义：

$$
\nabla p_t(x\mid y)
=
p_t(x\mid y)s_t(x\mid y)
\tag{13.10}
$$

**将式 (13.10) 代入式 (13.9)：**

$$
\nabla p_t(x)
=
\int
p_t(x\mid y)s_t(x\mid y)q(y)\,dy
\tag{13.11}
$$

两侧除以：

$$
p_t(x)
$$

得到：

$$
\frac{\nabla p_t(x)}{p_t(x)}
=
\int
s_t(x\mid y)
\frac{p_t(x\mid y)q(y)}{p_t(x)}
dy
\tag{13.12}
$$

根据边缘 Score 定义：

$$
s_t(x)=\frac{\nabla p_t(x)}{p_t(x)}
\tag{13.13}
$$

根据贝叶斯公式：

$$
p_t(y\mid x)
=
\frac{p_t(x\mid y)q(y)}{p_t(x)}
\tag{13.14}
$$

**将式 (13.13) 和式 (13.14) 代入式 (13.12)：**

$$
s_t(x)
=
\int
s_t(x\mid y)p_t(y\mid x)\,dy
\tag{13.15}
$$

因此：

$$
\boxed{
s_t(x)
=
\mathbb E[
s_t(x\mid Y)\mid X_t=x
]
}
\tag{13.16}
$$

---

## 13.3 边缘 Score 与边缘速度的比较

边缘 Score：

$$
\boxed{
s_t(x)
=
\mathbb E[
s_t(x\mid Y)\mid X_t=x
]
}
\tag{13.17}
$$

边缘速度：

$$
\boxed{
u_t(x)
=
\mathbb E[
u_t(x\mid Y)\mid X_t=x
]
}
\tag{13.18}
$$

两者都表现为贝叶斯后验平均。

但推导依据不同。

**Score：**

从概率密度的空间梯度出发：

$$
\nabla p=p\nabla\log p
$$

边缘 Score 由边缘概率密度唯一确定。

**Velocity：**

从连续性方程出发：

$$
\partial_tp+\nabla\cdot(pu)=0
$$

通过混合条件概率流量构造边缘速度。

因此，边缘速度不一定唯一。

---

## 13.4 DDPM 与 FM 的 MSE 训练

DDPM 常用噪声预测损失：

$$
\boxed{
\mathcal L_{\mathrm{DDPM}}
=
\mathbb E
\|\epsilon_\theta(X_t,t)-Z\|^2
}
\tag{13.19}
$$

根据 MSE 条件期望性质：

$$
\boxed{
\epsilon_\theta^*(x,t)
=
\mathbb E[Z\mid X_t=x]
}
\tag{13.20}
$$

根据式 (13.7) 和式 (13.16)：

$$
\boxed{
s_t(x)
=
-\frac{\mathbb E[Z\mid X_t=x]}{\sigma_t}
}
\tag{13.21}
$$

**将式 (13.20) 代入式 (13.21)：**

$$
\boxed{
s_t(x)
=
-\frac{\epsilon_\theta^*(x,t)}{\sigma_t}
}
\tag{13.22}
$$

Flow Matching 使用：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E
\|v_\theta(X_t,t)-U\|^2
}
\tag{13.23}
$$

其最优解：

$$
\boxed{
v_\theta^*(x,t)
=
\mathbb E[U\mid X_t=x]
}
\tag{13.24}
$$

因此二者共同利用：

$$
\boxed{
\text{MSE 最优解是条件期望}
}
\tag{13.25}
$$

实现从条件监督到边缘场的学习。

---

# 第十四章：线性 FM 中速度场与 Score 的转换

## 14.1 线性概率路径

定义：

$$
\boxed{
X_t=(1-t)Z+tY
}
\tag{14.1}
$$

其中：

$$
Z\sim\mathcal N(0,I)
$$

$$
Y\sim q
$$

并假设 $Z,Y$ 独立。

---

## 14.2 边缘 Score

固定：

$$
Y=y
$$

得到：

$$
p_t(x\mid y)
=
\mathcal N(x;ty,(1-t)^2I)
\tag{14.2}
$$

根据高斯条件 Score 公式 (13.5)：

$$
s_t(x\mid y)
=
-\frac{x-ty}{(1-t)^2}
\tag{14.3}
$$

根据式 (14.1)：

$$
x-ty=(1-t)z
\tag{14.4}
$$

**将式 (14.4) 代入式 (14.3)：**

$$
\boxed{
s_t(x\mid y)
=
-\frac z{1-t}
}
\tag{14.5}
$$

对条件 Score 进行后验平均：

$$
\boxed{
s_t(x)
=
-\frac{\mathbb E[Z\mid X_t=x]}{1-t}
}
\tag{14.6}
$$

---

## 14.3 边缘速度

根据式 (14.1)：

$$
X_t=(1-t)Z+tY
$$

对条件 $X_t=x$ 取期望：

$$
x
=
(1-t)\mathbb E[Z\mid X_t=x]
+
t\mathbb E[Y\mid X_t=x]
\tag{14.7}
$$

边缘速度：

$$
u_t(x)
=
\mathbb E[Y-Z\mid X_t=x]
\tag{14.8}
$$

定义：

$$
m_Z=\mathbb E[Z\mid X_t=x]
\tag{14.9}
$$

$$
m_Y=\mathbb E[Y\mid X_t=x]
\tag{14.10}
$$

则式 (14.7) 为：

$$
x=(1-t)m_Z+tm_Y
\tag{14.11}
$$

式 (14.8) 为：

$$
u_t=m_Y-m_Z
\tag{14.12}
$$

根据式 (14.12)：

$$
m_Y=u_t+m_Z
\tag{14.13}
$$

**将式 (14.13) 代入式 (14.11)：**

$$
x=(1-t)m_Z+t(u_t+m_Z)
$$

展开：

$$
x=(1-t)m_Z+tu_t+tm_Z
$$

合并：

$$
x=m_Z+tu_t
$$

因此：

$$
\boxed{
m_Z=x-tu_t
}
\tag{14.14}
$$

---

## 14.4 速度与 Score 的转换

根据式 (14.6)：

$$
s_t(x)
=
-\frac{m_Z}{1-t}
\tag{14.15}
$$

**将式 (14.14) 代入式 (14.15)：**

$$
s_t(x)
=
-\frac{x-tu_t(x)}{1-t}
$$

整理：

$$
\boxed{
s_t(x)
=
\frac{tu_t(x)-x}{1-t}
}
\tag{14.16}
$$

将式 (14.16) 移项：

$$
(1-t)s_t(x)=tu_t(x)-x
$$

得到：

$$
tu_t(x)=x+(1-t)s_t(x)
$$

因此：

$$
\boxed{
u_t(x)
=
\frac{x+(1-t)s_t(x)}{t}
}
\tag{14.17}
$$

其中：

- 式 (14.16) 适用于 $t<1$。
- 式 (14.17) 适用于 $t>0$。
- 端点需要单独处理或取适当极限。

这说明在特定线性高斯概率路径下，边缘速度和边缘 Score 可以互相转换。

但这种关系依赖概率路径结构，不能直接推广到任意路径。

---

# 第十五章：Simulation-free

## 15.1 Simulation-free 的定义

Flow Matching 的一个重要特点是：

$$
\boxed{
\text{训练阶段不需要数值求解生成 ODE}
}
\tag{15.1}
$$

以线性 FM 为例：

$$
X_t=(1-t)Z+tY
\tag{15.2}
$$

$$
U=Y-Z
\tag{15.3}
$$

训练时：

$$
Y\sim q
$$

$$
Z\sim\mathcal N(0,I)
$$

$$
t\sim U(0,1)
$$

可以直接计算：

$$
X_t
$$

和：

$$
U
$$

因此不需要逐步模拟：

$$
X_0\rightarrow X_{\Delta t}
\rightarrow X_{2\Delta t}
\rightarrow\cdots\rightarrow X_t
$$

---

## 15.2 Simulation-free 与一步加噪的关系

Simulation-free 和 DDPM 的直接加噪训练具有相似之处。

DDPM 可以直接构造：

$$
X_t=\alpha_tY+\sigma_tZ
\tag{15.4}
$$

FM 可以直接构造：

$$
X_t=(1-t)Z+tY
\tag{15.5}
$$

两者都不需要在训练阶段逐步模拟完整生成过程。

但是：

$$
\boxed{
\text{Simulation-free}
\neq
\text{推理一步生成}
}
\tag{15.6}
$$

Simulation-free 只说明训练时可以直接采样中间状态。

推理阶段仍然需要求解生成 ODE。

---

# 第十六章：为什么推理需要多步积分？

## 16.1 ODE 的精确积分形式

推理阶段：

$$
\boxed{
\frac{dX_t}{dt}=v_\theta(X_t,t)
}
\tag{16.1}
$$

根据微积分基本定理：

$$
\boxed{
X_1
=
X_0+
\int_0^1v_\theta(X_t,t)\,dt
}
\tag{16.2}
$$

这里的积分包含整条生成轨迹上的速度。

如果直接使用一步 Euler：

$$
\boxed{
\widehat X_1
=
X_0+v_\theta(X_0,0)
}
\tag{16.3}
$$

实际上是使用初始速度近似整个积分：

$$
\int_0^1v_\theta(X_t,t)\,dt
\approx
v_\theta(X_0,0)
\tag{16.4}
$$

当速度沿轨迹发生变化时，这种近似通常会产生误差。

---

## 16.2 Euler 积分

将时间划分为：

$$
0=t_0<t_1<\cdots<t_N=1
$$

设：

$$
h=t_{k+1}-t_k
$$

Euler 更新：

$$
\boxed{
X_{k+1}
=
X_k+h\,v_\theta(X_k,t_k)
}
\tag{16.5}
$$

每步需要一次网络评估。

---

## 16.3 Midpoint 积分

首先计算当前位置速度：

$$
k_1=v_\theta(X_k,t_k)
\tag{16.6}
$$

预测中点：

$$
X_{k+\frac12}
=
X_k+\frac h2k_1
\tag{16.7}
$$

计算中点速度：

$$
k_2=
v_\theta
\left(
X_{k+\frac12},
t_k+\frac h2
\right)
\tag{16.8}
$$

最终更新：

$$
\boxed{
X_{k+1}=X_k+hk_2
}
\tag{16.9}
$$

每步需要两次网络评估。

---

## 16.4 为什么 OT Path 也需要多步？

OT 条件路径：

$$
X_t=(1-t)Z+tY
$$

对应条件速度：

$$
Y-Z
$$

确实是恒定的。

但是模型最终学习的是：

$$
u_t(x)
=
\mathbb E[Y-Z\mid X_t=x]
$$

而不是针对某一对固定 $(Z,Y)$ 的速度。

根据第十章的推导，边缘速度通常依赖时间和空间位置。

因此：

$$
\boxed{
\text{条件速度恒定}
\not\Rightarrow
\text{边缘速度恒定}
}
\tag{16.10}
$$

所以，即使使用 OT-CFM 训练，推理时通常仍然需要多步数值积分。

---

# 第十七章：Flow Matching 与自动驾驶轨迹规划

## 17.1 条件轨迹生成任务

假设当前场景条件为：

$$
C=c
\tag{17.1}
$$

其中包括：

- 自车历史状态。
- 周围交通参与者。
- 道路拓扑。
- 地图信息。
- 导航信息。
- 交通规则。

真实未来轨迹：

$$
Y\in\mathbb R^{T\times D}
\tag{17.2}
$$

目标是学习：

$$
\boxed{
p(Y\mid C=c)
}
\tag{17.3}
$$

---

## 17.2 轨迹 Flow Matching

定义：

$$
Z\sim\mathcal N(0,I)
\tag{17.4}
$$

构造：

$$
\boxed{
X_t=(1-t)Z+tY
}
\tag{17.5}
$$

条件速度标签：

$$
\boxed{
U=Y-Z
}
\tag{17.6}
$$

训练网络：

$$
v_\theta(X_t,t,c)
\tag{17.7}
$$

损失：

$$
\boxed{
\mathcal L
=
\mathbb E_{t,Y,Z,C}
\left[
\|v_\theta(X_t,t,C)-(Y-Z)\|^2
\right]
}
\tag{17.8}
$$

这里存在两种不同的条件。

### 第一种：CFM 的条件

$$
p_t(x\mid Y=y)
$$

表示固定真实数据样本后构造的条件概率路径。

### 第二种：场景条件

$$
p(Y\mid C=c)
$$

表示给定驾驶场景时的轨迹条件分布。

两种条件可以同时存在。

---

## 17.3 条件轨迹规划的边缘速度

考虑给定场景条件：

$$
C=c
$$

时的概率路径：

$$
p_t(x\mid c)
\tag{17.9}
$$

根据全概率公式：

$$
p_t(x\mid c)
=
\int
p_t(x\mid y,c)q(y\mid c)\,dy
\tag{17.10}
$$

如果噪声与场景、真实轨迹独立，并使用线性路径：

$$
X_t=(1-t)Z+tY
$$

那么理论最优速度为：

$$
\boxed{
u_t(x,c)
=
\mathbb E[
Y-Z\mid X_t=x,C=c
]
}
\tag{17.11}
$$

因此，网络通过 MSE 学习的是给定场景和中间轨迹状态后的条件期望速度。

---

## 17.4 为什么确定性 ODE 能够生成多模态轨迹？

推理时：

$$
Z\sim\mathcal N(0,I)
\tag{17.12}
$$

通过 ODE：

$$
\boxed{
\frac{dX_t}{dt}
=
v_\theta(X_t,t,c)
}
\tag{17.13}
$$

得到：

$$
Y_{\mathrm{gen}}
=
\phi_1(Z;c)
\tag{17.14}
$$

虽然：

$$
v_\theta
$$

是确定性函数，但初始噪声：

$$
Z
$$

具有随机性。

因此不同初始噪声可以产生不同生成结果：

$$
Z_1\rightarrow Y_1
$$

$$
Z_2\rightarrow Y_2
$$

$$
Z_3\rightarrow Y_3
$$

所以：

$$
\boxed{
\text{随机初始噪声}
+
\text{确定性流映射}
\rightarrow
\text{随机生成分布}
}
\tag{17.15}
$$

但模型是否能够覆盖真实驾驶场景中的合理多模态轨迹，仍然取决于数据覆盖、模型容量、条件编码和训练质量。

---

## 17.5 单场景只有一条专家轨迹是否可以训练？

可以。

训练数据通常是：

$$
(c_i,y_i)
\tag{17.16}
$$

对于同一条专家轨迹：

$$
y_i
$$

可以采样不同的：

$$
Z,t
$$

构造：

$$
X_t=(1-t)Z+ty_i
\tag{17.17}
$$

并计算监督目标：

$$
U=y_i-Z
\tag{17.18}
$$

因此不要求每个场景具有多条专家轨迹。

但需要注意：

**能够进行训练，不意味着仅凭一条专家轨迹就能识别该场景下所有合理的驾驶模式。**

条件分布的多模态学习仍然依赖不同场景之间的统计规律以及训练数据的覆盖程度。

---

# 第十八章：论文实验与方法特点

## 18.1 论文实验的主要比较对象

论文在图像生成任务中比较了不同概率路径及训练方法，包括：

- 基于扩散概率路径的 FM。
- 基于条件 OT 概率路径的 FM。
- DDPM 等扩散模型。
- Score Matching 相关方法。

论文主要考察：

1. 生成样本质量。
2. 训练效果。
3. ODE 采样效率。
4. 不同概率路径对生成动力学的影响。

论文报告，采用 OT 条件路径的 Flow Matching 在其部分图像生成实验中取得了较好的生成质量与采样效率。

但这不意味着 OT Path 在所有生成任务中都必然优于扩散概率路径。

---

## 18.2 OT Path 的主要特点

OT 条件路径：

$$
X_t=(1-t)Z+tY
$$

具有：

$$
\dot X_t=Y-Z
$$

$$
\ddot X_t=0
$$

因此其条件监督目标简单。

相比某些扩散概率路径，这种路径不需要复杂的时间相关系数来计算训练目标。

但边缘速度仍然是：

$$
u_t(x)
=
\mathbb E[Y-Z\mid X_t=x]
$$

所以最终生成动力学仍然可能复杂。

---

## 18.3 Flow Matching 的主要优势与限制

| 方面 | 特点 |
|---|---|
| 训练 | 不需要在每次训练迭代中求解生成 ODE |
| 监督 | 使用可解析的条件速度 |
| 概率路径 | 可以选择不同条件概率路径 |
| 推理 | 通过 ODE 生成 |
| 似然 | CNF 框架允许计算似然 |
| 多模态 | 可以通过随机初始噪声生成不同样本 |
| 采样效率 | 与概率路径、速度场结构及求解器有关 |
| 理论限制 | 条件路径简单不保证边缘轨迹简单 |
| 数据限制 | 生成分布的质量仍依赖训练数据覆盖 |

---

# 第十九章：三个核心定理的统一理解

## 19.1 定理一：条件速度构造边缘速度

已知条件概率路径：

$$
p_t(x\mid y)
$$

及其条件速度：

$$
u_t(x\mid y)
$$

满足条件连续性方程。

那么可以构造：

$$
\boxed{
u_t(x)
=
\mathbb E[
u_t(x\mid Y)\mid X_t=x
]
}
\tag{19.1}
$$

作为能够生成边缘概率路径的合法速度场。

其数学依据是：

$$
\boxed{
\text{条件概率流量可以加权混合}
}
$$

---

## 19.2 定理二：CFM 与 FM 梯度相同

$$
\boxed{
\nabla_\theta\mathcal L_{\mathrm{CFM}}
=
\nabla_\theta\mathcal L_{\mathrm{FM}}
}
\tag{19.2}
$$

因此可以使用条件速度监督训练边缘速度场。

其数学依据是：

$$
\boxed{
\text{MSE 的条件期望性质}
}
$$

---

## 19.3 定理三：高斯条件速度具有解析形式

对于：

$$
X_t=\sigma_t(y)Z+\mu_t(y)
$$

条件速度为：

$$
\boxed{
u_t(x\mid y)
=
\frac{\dot\sigma_t(y)}{\sigma_t(y)}
[x-\mu_t(y)]
+
\dot\mu_t(y)
}
\tag{19.3}
$$

其数学依据是：

$$
\boxed{
\text{高斯仿射变换及其时间导数}
}
$$

---

# 第二十章：完整知识框架

| 模块 | 核心公式 | 主要作用 |
|---|---|---|
| 变量替换 | 式 (2.11) | 描述随机变量映射后的密度变化 |
| Pushforward | 式 (2.18) | 描述概率分布的确定性运输 |
| 连续性方程 | 式 (3.12) | 联系概率密度与速度场 |
| CNF | 式 (4.1) | 定义连续生成过程 |
| 瞬时变量替换 | 式 (4.15) | 计算 CNF 密度变化 |
| FM | 式 (4.20) | 直接监督边缘速度 |
| 条件概率路径 | 式 (5.2) | 构造易计算的训练路径 |
| 边缘概率路径 | 式 (5.6) | 混合条件概率路径 |
| 定理一 | 式 (6.17) | 从条件速度构造边缘速度 |
| CFM | 式 (7.2) | 使用条件速度训练 |
| 定理二 | 式 (7.30) | 证明 FM 与 CFM 梯度等价 |
| 高斯条件路径 | 式 (8.1) | 解析构造训练状态 |
| 定理三 | 式 (8.12) | 解析计算条件速度 |
| OT Path | 式 (9.12) | 构造条件直线路径 |
| OT-CFM | 式 (9.25) | 最终训练损失 |
| 边缘轨迹 | 式 (10.33) | 解释边缘速度的非恒定性 |
| VP Path | 式 (11.16) | 扩散路径的条件速度 |
| Probability Flow ODE | 式 (12.11) | 联系 SDE 与确定性 ODE |
| Score Matching | 式 (13.16) | 条件 Score 的边缘化 |
| Score–Velocity 转换 | 式 (14.16) | 特定路径下的参数化转换 |
| Simulation-free | 式 (15.1) | 训练无需 ODE 模拟 |
| ODE Sampling | 式 (16.1) | 推理生成 |
| 条件轨迹规划 | 式 (17.11) | 给定场景学习边缘速度 |

---

# 最终总结

Flow Matching 的完整理论逻辑为：

**第一步：定义生成过程。**

通过 CNF：

$$
\boxed{
\dot X_t=v_\theta(X_t,t)
}
$$

将初始高斯分布运输到真实数据分布。

**第二步：构造目标概率路径。**

定义：

$$
\boxed{
p_t(x)
}
$$

描述概率分布随时间的演化。

**第三步：构造条件概率路径。**

固定真实数据：

$$
Y=y
$$

构造：

$$
\boxed{
p_t(x\mid y)
}
$$

使其具有容易计算的条件速度。

**第四步：解析求解条件速度。**

对于高斯仿射路径：

$$
X_t=\sigma_t(y)Z+\mu_t(y)
$$

得到：

$$
\boxed{
u_t(x\mid y)
=
\frac{\dot\sigma_t(y)}{\sigma_t(y)}
[x-\mu_t(y)]
+
\dot\mu_t(y)
}
$$

**第五步：通过连续性方程构造边缘速度。**

$$
\boxed{
u_t(x)
=
\mathbb E[
u_t(x\mid Y)\mid X_t=x
]
}
$$

**第六步：使用 CFM 训练。**

$$
\boxed{
\mathcal L_{\mathrm{CFM}}
=
\mathbb E
\|v_\theta(X_t,t)-u_t(X_t\mid Y)\|^2
}
$$

由于：

$$
\boxed{
\nabla_\theta\mathcal L_{\mathrm{CFM}}
=
\nabla_\theta\mathcal L_{\mathrm{FM}}
}
$$

因此无需显式计算边缘速度。

**第七步：通过 ODE 推理生成。**

$$
\boxed{
X_1
=
X_0+
\int_0^1v_\theta(X_t,t)\,dt
}
$$

最终生成目标数据。

整篇论文最重要的理论区别是：

$$
\boxed{
\text{条件概率路径}
\neq
\text{边缘 ODE 的样本轨迹}
}
$$

条件路径可以是匀速直线，但边缘速度场通常依赖时间和空间位置，因此边缘 ODE 的轨迹可能弯曲。

这也解释了为什么 Flow Matching 即使采用恒定条件速度作为训练目标，推理时仍然通常需要多步数值积分。

**Flow Matching 的核心贡献是：利用简单、可解析的条件概率路径和条件速度监督，训练能够生成复杂边缘概率路径的 Continuous Normalizing Flow。**